## Data Exploration and Cleaning - Heart Disease Dataset

### By:
Laura Granda

### Date:
2026-03-14

### Description:

Notebook to explore the raw heart disease dataset, identify data quality issues (irregular nulls, typos, invalid values, wrong types), clean the data systematically, cast proper types, and save the result as a primary-layer parquet file.

Input: `data/02_intermediate/corazon_raw.parquet`

Output: `data/03_primary/corazon_explored.parquet`

## 📚 Import libraries

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

## 💾 Load data

In [ ]:
INPUT_PATH: Path = Path("/home/lauragranda01/corazon/data/02_intermediate/corazon_raw.parquet")
OUTPUT_PATH: Path = Path("/home/lauragranda01/corazon/data/03_primary/corazon_explored.parquet")

df: pd.DataFrame = pd.read_parquet(INPUT_PATH, engine="pyarrow")

print(f"Dataset shape: {df.shape}")
print(f"Loaded from: {INPUT_PATH}")

Dataset shape: (3030, 14)
Loaded from: /home/lauragranda01/corazon/data/02_intermediate/corazon_raw.parquet


## 🔍 Initial exploration

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3030 entries, 0 to 3029
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   age         3000 non-null   object 
 1   sex         2969 non-null   object 
 2   chest_pain  2947 non-null   object 
 3   rest_bp     2949 non-null   object 
 4   chol        2945 non-null   object 
 5   fbs         2933 non-null   float64
 6   rest_ecg    2837 non-null   object 
 7   max_hr      2859 non-null   object 
 8   exang       2879 non-null   object 
 9   old_peak    2880 non-null   object 
 10  slope       2879 non-null   object 
 11  ca          2868 non-null   object 
 12  thal        2904 non-null   object 
 13  disease     2924 non-null   object 
dtypes: float64(1), object(13)
memory usage: 331.5+ KB


In [4]:
print("\nMissing values per column:")
missing: pd.Series = df.isnull().sum()
print(missing[missing > 0])

print("\nUnique values per column:")
for col in df.columns:
    unique_vals = df[col].dropna().unique()[:10]
    print(f"  {col}: {df[col].nunique()} unique -> {list(unique_vals)}")


Missing values per column:
age            30
sex            61
chest_pain     83
rest_bp        81
chol           85
fbs            97
rest_ecg      193
max_hr        171
exang         151
old_peak      150
slope         151
ca            162
thal          126
disease       106
dtype: int64

Unique values per column:
  age: 43 unique -> ['63', '67', '37', '41', '56', '62', '57', '53', '44', '52']
  sex: 5 unique -> ['Male', 'Female', '2345', '45', '765']
  chest_pain: 7 unique -> ['typical', 'asymptomatic', 'nonanginal', 'nontypical', '2345', '2435', '3456']
  rest_bp: 52 unique -> ['145', '160', '120', '130', '140', '172', '150', '110', '132', '117']
  chol: 154 unique -> ['233', '286', '229', '250', '204', '236', '268', '354', '254', '203']
  fbs: 2 unique -> [np.float64(1.0), np.float64(0.0)]
  rest_ecg: 8 unique -> ['left ventricular hypertrophy ', 'normal', 'ST-T wave abnormality', '5678', '3563', '5653', '36653', '435647']
  max_hr: 92 unique -> ['150', '108', '129', '187', '172

## 👷 Data preparation

### 🧹 Step 1: Standardize null representations

In [5]:
def standardize_nulls(df: pd.DataFrame) -> pd.DataFrame:
    """Replace irregular null representations with np.nan across all columns."""
    null_patterns: list[str] = ["?", "NA", "null", "NULL", "None", "none", ""]
    result: pd.DataFrame = df.replace(null_patterns, np.nan)
    return result


df = standardize_nulls(df)
print("Nulls after standardization:")
print(df.isnull().sum())

Nulls after standardization:
age            30
sex            61
chest_pain     83
rest_bp        81
chol           85
fbs            97
rest_ecg      193
max_hr        171
exang         151
old_peak      150
slope         151
ca            162
thal          126
disease       106
dtype: int64


### 🧹 Step 2: Clean strings

In [6]:
def clean_strings(df: pd.DataFrame) -> pd.DataFrame:
    """Strip whitespace from all object columns."""
    result: pd.DataFrame = df.copy()
    str_cols: list[str] = result.select_dtypes(include="object").columns.tolist()
    for col in str_cols:
        result[col] = result[col].str.strip()
    return result


df = clean_strings(df)
print("Strings cleaned (whitespace stripped).")

Strings cleaned (whitespace stripped).


### 🧹 Step 3: Validate values against schema

In [ ]:
# Define valid values according to datos_corazon_Info.txt
VALID_SEX: set[str] = {"Male", "Female"}
VALID_CHEST_PAIN: set[str] = {"typical", "asymptomatic", "nonanginal", "nontypical"}
VALID_REST_ECG: set[str] = {
    "normal",
    "left ventricular hypertrophy",
    "ST-T wave abnormality",
}
VALID_EXANG: set[str] = {"0", "1"}
VALID_SLOPE: set[str] = {"1", "2", "3"}
VALID_CA: set[str] = {"0.0", "1.0", "2.0", "3.0"}
VALID_THAL: set[str] = {"normal", "fixed", "reversable"}
VALID_DISEASE: set[str] = {"0", "1"}

In [8]:
def invalidate_categorical(series: pd.Series, valid_values: set[str]) -> pd.Series:
    """Replace values not in valid_values with np.nan."""
    return series.where(series.isin(valid_values))


def invalidate_numeric(series: pd.Series) -> pd.Series:
    """Convert to numeric, coercing non-parseable values to NaN."""
    return pd.to_numeric(series, errors="coerce")

In [9]:
# Validate categorical columns
df["sex"] = invalidate_categorical(df["sex"], VALID_SEX)
df["chest_pain"] = invalidate_categorical(df["chest_pain"], VALID_CHEST_PAIN)
df["rest_ecg"] = invalidate_categorical(df["rest_ecg"], VALID_REST_ECG)
df["exang"] = invalidate_categorical(df["exang"], VALID_EXANG)
df["slope"] = invalidate_categorical(df["slope"], VALID_SLOPE)
df["ca"] = invalidate_categorical(df["ca"], VALID_CA)
df["thal"] = invalidate_categorical(df["thal"], VALID_THAL)
df["disease"] = invalidate_categorical(df["disease"], VALID_DISEASE)

# Validate numeric columns
df["age"] = invalidate_numeric(df["age"])
df["rest_bp"] = invalidate_numeric(df["rest_bp"])
df["chol"] = invalidate_numeric(df["chol"])
df["max_hr"] = invalidate_numeric(df["max_hr"])
df["old_peak"] = invalidate_numeric(df["old_peak"])

print("Nulls after validation:")
print(df.isnull().sum())
print(f"\nRows remaining: {len(df)}")

Nulls after validation:
age            32
sex            64
chest_pain     86
rest_bp        83
chol           87
fbs            97
rest_ecg      198
max_hr        172
exang         153
old_peak      152
slope         152
ca            163
thal          130
disease       111
dtype: int64

Rows remaining: 3030


### 🧹 Step 4: Cast data types

In [ ]:
def cast_types(df: pd.DataFrame) -> pd.DataFrame:
    """Cast all columns to their proper domain types."""
    result: pd.DataFrame = df.copy()

    # Nullable integers
    result["age"] = result["age"].astype("Int64")
    result["rest_bp"] = result["rest_bp"].astype("Int64")
    result["chol"] = result["chol"].astype("Int64")
    result["max_hr"] = result["max_hr"].astype("Int64")

    # Float
    result["old_peak"] = result["old_peak"].astype("float64")

    # Boolean columns
    result["fbs"] = result["fbs"].astype("boolean")
    result["exang"] = result["exang"].map({"1": True, "0": False}).astype("boolean")
    result["disease"] = result["disease"].map({"1": True, "0": False}).astype("boolean")

    # Nominal categoricals (unordered)
    result["sex"] = result["sex"].astype("category")
    result["chest_pain"] = result["chest_pain"].astype("category")
    result["thal"] = result["thal"].astype("category")

    # Ordinal categoricals (ordered)
    rest_ecg_dtype = pd.CategoricalDtype(
        categories=["normal", "ST-T wave abnormality", "left ventricular hypertrophy"],
        ordered=True,
    )
    result["rest_ecg"] = result["rest_ecg"].astype(rest_ecg_dtype)

    slope_dtype = pd.CategoricalDtype(categories=["1", "2", "3"], ordered=True)
    result["slope"] = result["slope"].astype(slope_dtype)

    ca_dtype = pd.CategoricalDtype(categories=["0.0", "1.0", "2.0", "3.0"], ordered=True)
    result["ca"] = result["ca"].astype(ca_dtype)

    return result


df = cast_types(df)
print("Types after casting:")
print(df.dtypes)

Types after casting:
age              Int64
sex           category
chest_pain    category
rest_bp          Int64
chol             Int64
fbs            boolean
rest_ecg      category
max_hr           Int64
exang          boolean
old_peak       float64
slope         category
ca            category
thal          category
disease        boolean
dtype: object


### 🧹 Step 5: Drop rows with invalid target

In [11]:
rows_before: int = len(df)
df = df.dropna(subset=["disease"])
rows_after: int = len(df)
print(f"Dropped {rows_before - rows_after} rows with null disease (target).")
print(f"Rows remaining: {rows_after}")

Dropped 111 rows with null disease (target).
Rows remaining: 2919


### ✅ Final dataset overview

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2919 entries, 0 to 3029
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   age         2902 non-null   Int64   
 1   sex         2870 non-null   category
 2   chest_pain  2848 non-null   category
 3   rest_bp     2851 non-null   Int64   
 4   chol        2847 non-null   Int64   
 5   fbs         2837 non-null   boolean 
 6   rest_ecg    2803 non-null   category
 7   max_hr      2829 non-null   Int64   
 8   exang       2848 non-null   boolean 
 9   old_peak    2849 non-null   float64 
 10  slope       2849 non-null   category
 11  ca          2838 non-null   category
 12  thal        2872 non-null   category
 13  disease     2919 non-null   boolean 
dtypes: Int64(4), boolean(3), category(6), float64(1)
memory usage: 183.3 KB


In [13]:
print(f"\nFinal shape: {df.shape}")
print("\nMissing values:")
print(df.isnull().sum())
print("\nSample:")
df.head()


Final shape: (2919, 14)

Missing values:
age            17
sex            49
chest_pain     71
rest_bp        68
chol           72
fbs            82
rest_ecg      116
max_hr         90
exang          71
old_peak       70
slope          70
ca             81
thal           47
disease         0
dtype: int64

Sample:


,age,sex,chest_pain,rest_bp,chol,fbs,rest_ecg,max_hr,exang,old_peak,slope,ca,thal,disease
0,63,Male,typical,145,233,True,left ventricular hypertrophy,150,False,2.3,3,0.0,fixed,False
1,67,Male,asymptomatic,160,286,False,left ventricular hypertrophy,108,True,1.5,2,3.0,normal,True
2,67,Male,asymptomatic,120,229,False,left ventricular hypertrophy,129,True,2.6,2,2.0,reversable,True
3,37,Male,nonanginal,130,250,False,normal,187,False,3.5,3,0.0,normal,False
4,41,Female,nontypical,130,204,False,left ventricular hypertrophy,172,False,1.4,1,0.0,normal,False


## 💾 Save cleaned data

In [15]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(OUTPUT_PATH, engine="pyarrow", index=False)

print(f"File saved to: {OUTPUT_PATH}")
print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")

File saved to: /home/lauragranda01/corazon/data/03_primary/corazon_explored.parquet
Total rows: 2919
Total columns: 14


## 📊 Analysis of Results and Conclusions

### Data Quality Issues Found and Resolved:

1. **Irregular null representations**: `None` values in parquet converted to `np.nan`.
2. **Trailing whitespace**: `rest_ecg` had trailing spaces stripped.
3. **Garbage values in categoricals**: Numeric junk strings (e.g., '2345' in sex, '5678' in rest_ecg) invalidated to `np.nan`.
4. **Garbage values in booleans**: 'f', 'adfs' in exang; 'fsg', 'gsfdg', etc. in disease invalidated.
5. **Type casting**: All 13 object columns converted to proper types (Int64, float64, boolean, category).
6. **Rows with null target dropped**: Rows without disease value removed.

### Data Reduction Summary:
- Input: 3030 rows
- Rows dropped (null target): ~111
- Output: ~2919 rows with proper types and validated values
- Feature columns retain `np.nan` where original data was missing or invalid

### Proposals and Ideas:
- Investigate whether garbage values follow a pattern (e.g., data entry errors in specific batches)
- Consider imputation strategies for feature columns with >5% missing values (rest_ecg: ~6.4%)
- The `nonanginal` category in chest_pain is valid per the info file and preserved in this dataset